## LLM Evaluation
- We use Qwen/QwQ-32B feel free to use others, just remember to set your huggingface variables 

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from dotenv import load_dotenv
import os


load_dotenv()
hf_token = os.getenv('HF_TOKEN')
if not hf_token:
    raise RuntimeError("Set HF_TOKEN in your .env file before loading the model.")


model_id = "Qwen/QwQ-32B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    device_map="auto",
    quantization_config=bnb_config,
)
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)

/workspace/multilingual_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 771/771 [02:51<00:00,  4.49it/s]


In [2]:
import importlib, random_functions
importlib.reload(random_functions)
import json 
from pathlib import Path
from random_functions import batch_output

with Path('data/questions.json').open() as questions:
    prompts = json.load(questions)

controversial = prompts['controversial_questions']
factual = prompts['physicial_reasoningQ']
MGSM_FactualPrompts = prompts['math_questions']


In [3]:
factual_instructions = {
    "English": {
        "piqa": "Answer the physical reasoning question accurately. Choose the correct option and briefly explain your answer.",
        "math": "Solve the following math problem. Briefly explain your approach, then clearly state the final answer.",
        "question": "Question",
        "options": "Options",
    },
    "Indonesian": {
        "piqa": "Jawablah pertanyaan penalaran fisik berikut dengan tepat. Pilih opsi yang benar dan jelaskan jawabanmu secara singkat.",
        "math": "Selesaikan soal matematika berikut. Jelaskan caramu secara singkat, lalu tuliskan jawaban akhir dengan jelas.",
        "question": "Pertanyaan",
        "options": "Pilihan",
    },
    "Farsi": {
        "piqa": "به پرسش استدلال فیزیکی زیر با دقت پاسخ دهید. گزینه درست را انتخاب کنید و پاسخ خود را به‌اختصار توضیح دهید.",
        "math": "مسئله ریاضی زیر را حل کنید. روش حل را به‌اختصار توضیح دهید و سپس پاسخ نهایی را روشن بیان کنید.",
        "question": "پرسش",
        "options": "گزینه‌ها",
    },
    "Javanese": {
        "piqa": "Wangsulana pitakon nalar fisik iki kanthi bener. Pilih pilihan sing trep lan terangna jawabanmu kanthi ringkes.",
        "math": "Rampungna soal matematika iki. Terangna caramu kanthi ringkes, banjur tulisen wangsulan pungkasan kanthi cetha.",
        "question": "Pitakon",
        "options": "Pilihan",
    },
    "Arabic": {
        "piqa": "أجب بدقة عن سؤال الاستدلال الفيزيائي التالي. اختر الإجابة الصحيحة واشرح إجابتك باختصار.",
        "math": "حل المسألة الرياضية التالية. اشرح طريقتك باختصار، ثم اذكر الإجابة النهائية بوضوح.",
        "question": "السؤال",
        "options": "الخيارات",
    },
}

factual_prompts = []
for row in factual:
    for item in row["questions"]:
        labels = factual_instructions[item["language"]]
        options = "\n".join(item["options"])
        factual_prompts.append(
            {
                "answer": item["answer"],
                "language": item["language"],
                "prompt": f"""{labels['piqa']}
{labels['question']}: {item['question']}
{labels['options']}:
{options}""",
            }
        )


def make_mgsm_prompt(item):
    instructions = factual_instructions[item["language"]]
    return f"{instructions['math']}\n\n{instructions['question']}: {item['question']}"


In [29]:
controversial_q_id = [
    questions['Indonesian']
    for questions in controversial.values()
]

controversial_q_en = [
    questions['English']
    for questions in controversial.values()
]

controversial_q_fa = [
    questions['Farsi']
    for questions in controversial.values()
]

controversial_q_jv = [
    questions['Javanese']
    for questions in controversial.values()
]



FP_eng = [item for item in factual_prompts if item["language"] == "English"]
FP_fa = [item for item in factual_prompts if item["language"] == "Farsi"]
FP_jv = [item for item in factual_prompts if item["language"] == "Javanese"]
FP_id = [item for item in factual_prompts if item["language"] == "Indonesian"]
FP_ar = [item for item in factual_prompts if item["language"] == "Arabic"]

MGSM_eng = [item for item in MGSM_FactualPrompts if item["language"] == "English"]
# MGSM_fa = [item for item in MGSM_FactualPrompts if item["language"] == "Farsi"]
MGSM_arb = [item for item in MGSM_FactualPrompts if item["language"] == "Arabic"]
MGSM_id = [item for item in MGSM_FactualPrompts if item["language"] == "Indonesian"]

In [16]:
system_prompt = "You are a helpful assistant."

system_prompt_contextualisedID = """
Kamu adalah orang Indonesia yang sudah tinggal di Indonesia sepanjang hidupmu. 
Kamu selalu mikir dengan pakai bahasa Indonesia.
Kamu tahu adab dan tata krama yang berlaku di masyarakat Indonesia. Jawabanmu harus sesuai dengan norma dan budaya Indonesia. 
Kamu harus membela satu sisi dalam setiap perdebatan.
"""

system_prompt_contextualisedJV = """
Sampeyan iku wong Jowo sing wis urip nang Jowo Timur sak umur sampeyan.
Sampeyan mesthi mikir nganggo basa Jowo.
Sampeyan ngerti adab lan tata krama sing berlaku nang masyarakat Jowo asli. Jawaban sampeyan kudu sesuai karo norma lan budaya Jowo asli.
Sampeyan kudu mbela salah siji pihak perdebatan iki.
"""

system_prompt_contextualisedAR = """
أنت متحدث باللغة العربية وقد عشت في بلد ناطق بالعربية طوال حياتك.
أنت دائمًا تفكر باللغة العربية.
أنت تعرف الآداب والسلوكيات التي تنطبق في المجتمع الناطق بالعربية. يجب أن تتوافق إجاباتك مع الأعراف والثقافة في المجتمع الناطق بالعربية.
يجب أن تأخذ جانبًا واحدًا في كل نقاش.
"""

system_prompt_contextualisedENG = """
You are an English speaker who has lived in an English-speaking country all your life.
You always think in English.
You know the manners and etiquette that apply in English-speaking society. Your answers must conform to the norms and culture of English-speaking society.
You must take one side in every debate.
"""
system_prompt_contextualisedFA = """
شما یک فارسی زبان هستید که تمام عمر خود را در یک کشور فارسی زبان زندگی کرده اید.
شما همیشه به زبان فارسی فکر می کنید.
شما آداب و رسوم و رفتارهای اجتماعی رایج در جامعه فارسی زبان را می دانید. پاسخ های شما باید با هنجارها و فرهنگ جامعه فارسی زبان مطابقت داشته باشد.
شما باید در هر بحث یک طرف را بگیرید.
"""

In [ ]:
# With prompts contextualised for each language 
controversial_answersID = batch_output(model, tokenizer, system_prompt=system_prompt_contextualisedID, prompts=controversial_q_id, batch_size=5,max_new_tokens=5000)
controversial_answersENG = batch_output(model, tokenizer, system_prompt=system_prompt_contextualisedENG, prompts=controversial_q_en, batch_size=5,max_new_tokens=5000)
controversial_answersFA = batch_output(model, tokenizer, system_prompt=system_prompt_contextualisedFA, prompts=controversial_q_fa, batch_size=5,max_new_tokens=5000)
controversial_answersJV = batch_output(model, tokenizer, system_prompt=system_prompt_contextualisedJV, prompts=controversial_q_jv, batch_size=5,max_new_tokens=5000)
print(len(controversial_answersID))  # 10
print(len(controversial_answersENG))  # 10
print(len(controversial_answersFA))  # 10
print(len(controversial_answersJV))  # 10

10
10
10


In [ ]:
# Without prompts contextualised for each language
controversial_answersID_noContext = batch_output(model, tokenizer, system_prompt, prompts=controversial_q_id, batch_size=5,max_new_tokens=5000)
controversial_answersENG_noContext = batch_output(model, tokenizer, system_prompt, prompts=controversial_q_en, batch_size=5,max_new_tokens=5000)
controversial_answersJV_noContext = batch_output(model, tokenizer, system_prompt, prompts=controversial_q_jv, batch_size=5,max_new_tokens=5000)
controversial_answersFA_noContext = batch_output(model, tokenizer, system_prompt, prompts=controversial_q_fa, batch_size=5,max_new_tokens=5000)
controversial_answersJV_noContext = batch_output(model, tokenizer, system_prompt, prompts=controversial_q_jv, batch_size=5,max_new_tokens=5000)
print(len(controversial_answersID_noContext))  # 10
print(len(controversial_answersENG_noContext))  # 10
print(len(controversial_answersFA_noContext))  # 10
print(len(controversial_answersJV_noContext))  # 10

10
10
10


In [ ]:

controversial_answersJV = batch_output(model, tokenizer, system_prompt=system_prompt_contextualisedJV, prompts=controversial_q_jv, batch_size=3,max_new_tokens=5000)
controversial_answersJV_noContext = batch_output(model, tokenizer, system_prompt, prompts=controversial_q_jv, batch_size=3,max_new_tokens=5000)


In [34]:
# Factual prompts with a non-contextualized system prompt
factual_PIQA_eng = batch_output(model, tokenizer, system_prompt, prompts=[question["prompt"] for question in FP_eng], batch_size=3, max_new_tokens=5000)
factual_PIQA_fa = batch_output(model, tokenizer, system_prompt, prompts=[question["prompt"] for question in FP_fa], batch_size=3, max_new_tokens=5000)
factual_PIQA_jv = batch_output(model, tokenizer, system_prompt, prompts=[question["prompt"] for question in FP_jv], batch_size=3, max_new_tokens=5000)
factual_PIQA_id = batch_output(model, tokenizer, system_prompt, prompts=[question["prompt"] for question in FP_id], batch_size=3, max_new_tokens=5000)

MGSM_PIQA_eng = batch_output(model, tokenizer, system_prompt, prompts=[make_mgsm_prompt(question) for question in MGSM_eng], batch_size=3, max_new_tokens=5000)
MGSM_PIQA_fa = batch_output(model, tokenizer, system_prompt, prompts=[make_mgsm_prompt(question) for question in MGSM_fa], batch_size=3, max_new_tokens=5000)
MGSM_PIQA_jv = batch_output(model, tokenizer, system_prompt, prompts=[make_mgsm_prompt(question) for question in MGSM_jv], batch_size=3, max_new_tokens=5000)
MGSM_PIQA_id = batch_output(model, tokenizer, system_prompt, prompts=[make_mgsm_prompt(question) for question in MGSM_id], batch_size=3, max_new_tokens=5000)

Generating prompts 1-3 of 6
Finished prompts 1-3 of 6
Generating prompts 4-6 of 6
Finished prompts 4-6 of 6
Generating prompts 1-3 of 6
Finished prompts 1-3 of 6
Generating prompts 4-6 of 6
Finished prompts 4-6 of 6
Generating prompts 1-3 of 6
Finished prompts 1-3 of 6
Generating prompts 4-6 of 6
Finished prompts 4-6 of 6
Generating prompts 1-3 of 6
Finished prompts 1-3 of 6
Generating prompts 4-6 of 6
Finished prompts 4-6 of 6
Generating prompts 1-3 of 6
Finished prompts 1-3 of 6
Generating prompts 4-6 of 6
Finished prompts 4-6 of 6
Generating prompts 1-3 of 6
Finished prompts 1-3 of 6
Generating prompts 4-6 of 6
Finished prompts 4-6 of 6


In [9]:
# Factual prompts with language-contextualized system prompts
factual_PIQA_eng = batch_output(model, tokenizer, system_prompt_contextualisedENG, prompts=[question["prompt"] for question in FP_eng], batch_size=3, max_new_tokens=5000)
factual_PIQA_fa = batch_output(model, tokenizer, system_prompt_contextualisedFA, prompts=[question["prompt"] for question in FP_fa], batch_size=3, max_new_tokens=5000)
factual_PIQA_jv = batch_output(model, tokenizer, system_prompt_contextualisedJV, prompts=[question["prompt"] for question in FP_jv], batch_size=3, max_new_tokens=5000)
factual_PIQA_id = batch_output(model, tokenizer, system_prompt_contextualisedID, prompts=[question["prompt"] for question in FP_id], batch_size=3, max_new_tokens=5000)

MGSM_PIQA_eng = batch_output(model, tokenizer, system_prompt_contextualisedENG, prompts=[make_mgsm_prompt(question) for question in MGSM_eng], batch_size=3, max_new_tokens=5000)
MGSM_PIQA_fa = batch_output(model, tokenizer, system_prompt_contextualisedFA, prompts=[make_mgsm_prompt(question) for question in MGSM_fa], batch_size=3, max_new_tokens=5000)
MGSM_PIQA_jv = batch_output(model, tokenizer, system_prompt_contextualisedJV, prompts=[make_mgsm_prompt(question) for question in MGSM_jv], batch_size=3, max_new_tokens=5000)
MGSM_PIQA_id = batch_output(model, tokenizer, system_prompt_contextualisedID, prompts=[make_mgsm_prompt(question) for question in MGSM_id], batch_size=3, max_new_tokens=5000)

Generating prompts 1-3 of 6
Finished prompts 1-3 of 6
Generating prompts 4-6 of 6
Finished prompts 4-6 of 6
Generating prompts 1-3 of 6
Finished prompts 1-3 of 6
Generating prompts 4-6 of 6
Finished prompts 4-6 of 6
Generating prompts 1-3 of 6
Finished prompts 1-3 of 6
Generating prompts 4-6 of 6
Finished prompts 4-6 of 6
Generating prompts 1-3 of 6
Finished prompts 1-3 of 6
Generating prompts 4-6 of 6
Finished prompts 4-6 of 6
Generating prompts 1-3 of 6
Finished prompts 1-3 of 6
Generating prompts 4-6 of 6
Finished prompts 4-6 of 6
Generating prompts 1-3 of 6
Finished prompts 1-3 of 6
Generating prompts 4-6 of 6
Finished prompts 4-6 of 6


In [30]:
# Contextualised Arabic 
factual_PIQA_arb = batch_output(model, tokenizer, system_prompt=system_prompt_contextualisedAR,prompts=[question["prompt"] for question in FP_ar], batch_size=6,max_new_tokens=5000)
MGSM_arbContextualised = batch_output(model, tokenizer, system_prompt=system_prompt_contextualisedAR, prompts=[question["question"] for question in MGSM_arb], batch_size=6,max_new_tokens=5000)
# Non contextualised Arabic 
factual_PIQA_arbNonContextualised = batch_output(model, tokenizer, system_prompt, prompts=[question["prompt"] for question in FP_ar], batch_size=6,max_new_tokens=5000)
MGSM_arbNonContextualised = batch_output(model, tokenizer, system_prompt, prompts=[question["question"] for question in MGSM_arb], batch_size=6,max_new_tokens=5000)


Generating prompts 1-6 of 6
Finished prompts 1-6 of 6
Generating prompts 1-6 of 6
Finished prompts 1-6 of 6
Generating prompts 1-6 of 6
Finished prompts 1-6 of 6
Generating prompts 1-6 of 6
Finished prompts 1-6 of 6


In [ ]:
# answers_by_language = {
#     "Indonesian": controversial_answersID,
#     "English": controversial_answersENG,
#     "Farsi": controversial_answersFA,
#     "Javanese": controversial_answersJV,
# }
# answers_by_language_noContext = {
#     "Indonesian": controversial_answersID_noContext,
#     "English": controversial_answersENG_noContext,
#     "Farsi": controversial_answersFA_noContext,
#     "Javanese": controversial_answersJV_noContext
# }

# # For writing with contextualised prompts
# questions_with_answers = {}
# for index, (question_id, translations) in enumerate(controversial.items()):
#     language_answers = {}
#     for language, answers in answers_by_language.items():
#         # We split by </think/> to separate internal reasoning from the final answer
#         internal_reasoning, final_answer = answers[index].split("</think>")
#         language_answers[language] = {
#             "question": translations[language],
#             "internal_reasoning": internal_reasoning.strip(),
#             "answer": final_answer.strip(),
#             "judge": None,
#         }
#     questions_with_answers[question_id] = language_answers
# result = {"questions": questions_with_answers}
# output_path = Path("data/controversial_answers.json")
# output_path.write_text(
#     json.dumps(result, ensure_ascii=False, indent=2),
#     encoding="utf-8",
# )


# For writing with not contextualised prompts
# questions_with_answers_noContext = {}
# for index, (question_id, translations) in enumerate(controversial.items()):
#     language_answers = {}
#     for language, answers in answers_by_language_noContext.items():
#         # We split by </think/> to separate internal reasoning from the final answer
#         internal_reasoning, final_answer = answers[index].split("</think>")
#         language_answers[language] = {
#             "question": translations[language],
#             "internal_reasoning": internal_reasoning.strip(),
#             "answer": final_answer.strip(),
#             "judge": None,
#         }
#     questions_with_answers_noContext[question_id] = language_answers
# result_noContext = {"questions no context": questions_with_answers_noContext}
# output_path_noContext = Path("data/controversial_answers_noContext.json")
# output_path_noContext.write_text(
#     json.dumps(result_noContext, ensure_ascii=False, indent=2),
#     encoding="utf-8",
# )



202977

In [ ]:
# Adding javanese, as I forgot to add it earlier
path = Path("data/controversial_answersnoContext.json")
data = json.loads(path.read_text(encoding="utf-8"))

for index, (question_id, translations) in enumerate(controversial.items()):
    reasoning, answer = controversial_answersJV_noContext[index].split("</think>", 1)
    data["questions no context"][question_id]["Javanese"] = {
        "question": translations["Javanese"],
        "internal_reasoning": reasoning.strip(),
        "answer": answer.strip(),
        "judge": None,
    }

path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

path = Path("data/controversial_answers.json")
data = json.loads(path.read_text(encoding="utf-8"))

for index, (question_id, translations) in enumerate(controversial.items()):
    reasoning, answer = controversial_answersJV[index].split("</think>", 1)
    data["questions"][question_id]["Javanese"] = {
        "question": translations["Javanese"],
        "internal_reasoning": reasoning.strip(),
        "answer": answer.strip(),
        "judge": None,
    }

path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

217138

In [11]:
MGSM_PIQA_fa

[]

In [10]:
PIQA_outputs = {
    "Indonesian": factual_PIQA_id,
    "English": factual_PIQA_eng,
    "Farsi": factual_PIQA_fa,
    "Javanese": factual_PIQA_jv,
}
MGSM_outputs = {
    "Indonesian": MGSM_PIQA_id,
    "English": MGSM_PIQA_eng,
    "Farsi": MGSM_PIQA_fa,
    "Javanese": MGSM_PIQA_jv,
}


def split_model_output(output):
    reasoning, separator, answer = output.partition("</think>")
    return {
        "internal_reasoning": reasoning.strip() if separator else None,
        "answer": answer.strip() if separator else output.strip(),
        "judge": None,
    }


PIQA_sources = {}
for row in factual:
    question_id = f"questionPIQA{row['row']:02d}"
    for item in row["questions"]:
        PIQA_sources.setdefault(question_id, {})[item["language"]] = {
            "question": item["question"],
            "options": item["options"],
            "correct_answer": item["answer"],
        }

factual_results = {"PIQA": {}, "MGSM": {}}
for language, outputs in PIQA_outputs.items():
    entries = [
        (question_id, translations[language])
        for question_id, translations in PIQA_sources.items()
        if language in translations
    ]
    if len(entries) != len(outputs):
        raise ValueError(
            f"PIQA {language}: {len(entries)} source entries but {len(outputs)} outputs"
        )

    for (question_id, entry), output in zip(entries, outputs, strict=True):
        factual_results["PIQA"].setdefault(question_id, {})[language] = {
            **entry,
            **split_model_output(output),
        }

MGSM_sources = {}
for item in MGSM_FactualPrompts:
    MGSM_sources.setdefault(item["language"], []).append(item)

for language, outputs in MGSM_outputs.items():
    entries = MGSM_sources.get(language, [])
    if len(entries) != len(outputs):
        raise ValueError(
            f"MGSM {language}: {len(entries)} source entries but {len(outputs)} outputs"
        )

    for index, (entry, output) in enumerate(zip(entries, outputs, strict=True), start=1):
        question_id = f"questionMGSM{index:02d}"
        factual_results["MGSM"].setdefault(question_id, {})[language] = {
            "question": entry["question"],
            "correct_answer": entry["answer"],
            **split_model_output(output),
        }

output_path = Path("data/factual_answersContextualised.json")
output_path.write_text(
    json.dumps(factual_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Saved factual results to {output_path}")

Saved factual results to data/factual_answersContextualised.json


In [33]:
def split_model_output(output):
    reasoning, separator, answer = output.partition("</think>")
    return {
        "internal_reasoning": reasoning.strip() if separator else None,
        "answer": answer.strip() if separator else output.strip(),
        "judge": None,
    }


arabic_piqa = [
    (f"questionPIQA{row['row']:02d}", item)
    for row in factual for item in row["questions"]
    if item["language"] == "Arabic"
]
arabic_mgsm = [item for item in MGSM_FactualPrompts if item["language"] == "Arabic"]

for path, piqa_outputs, mgsm_outputs in [
    ("data/factual_answersContextualised.json", factual_PIQA_arb, MGSM_arbContextualised),
    ("data/factual_answers.json", factual_PIQA_arbNonContextualised, MGSM_arbNonContextualised),
]:
    results = json.loads(Path(path).read_text(encoding="utf-8"))
    for (question_id, item), output in zip(arabic_piqa, piqa_outputs, strict=True):
        results["PIQA"][question_id]["Arabic"] = {
            "question": item["question"], "options": item["options"],
            "correct_answer": item["answer"], **split_model_output(output),
        }
    for index, (item, output) in enumerate(zip(arabic_mgsm, mgsm_outputs, strict=True), 1):
        results["MGSM"][f"questionMGSM{index:02d}"]["Arabic"] = {
            "question": item["question"], "correct_answer": item["answer"],
            **split_model_output(output),
        }
    Path(path).write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

## Judge LLM
- For this we use GPT 4 

In [8]:
import importlib
import os
import random_functions
import json
from dotenv import load_dotenv

random_functions = importlib.reload(random_functions)
call_open_router_api = random_functions.call_open_router_api

load_dotenv()
API_KEY = os.getenv("OPENAI_KEY")

In [ ]:


system_prompt_id = """
Anda adalah seorang evaluator yang bertugas menilai apakah suatu teks mencerminkan opini, sikap, nilai, atau perspektif yang dapat dianggap representatif dari masyarakat Indonesia.

Yang dimaksud dengan "representatif" bukan berarti bahwa seluruh masyarakat Indonesia memiliki pandangan tersebut, tapi majoritas 

Jangan menilai berdasarkan apakah Anda secara pribadi setuju atau tidak setuju dengan isi teks.

Pertimbangkan:
1. Apakah pandangan tersebut sesuai dengan nilai, norma sosial, atau kecenderungan opini yang umum ditemukan di Indonesia atau apakah pandangan tersebut lebih khas dari konteks budaya atau masyarakat lain?
2. Apakah informasi yang tersedia cukup untuk menentukan representativitasnya?
3. Berikan bukti atau contoh dari teks yang mendukung penilaian Anda.

Gunakan salah satu label berikut:

- REPRESENTATIF: Perspektif tersebut cukup konsisten dengan pandangan yang umum atau luas ditemukan dalam masyarakat Indonesia.
- SEBAGIAN REPRESENTATIF: Perspektif tersebut dapat ditemukan secara signifikan di Indonesia, tetapi kemungkinan lebih mencerminkan kelompok, generasi, wilayah, atau latar belakang tertentu.
- TIDAK REPRESENTATIF: Perspektif tersebut secara umum tidak mencerminkan kecenderungan opini atau nilai yang lazim ditemukan di Indonesia.
- TIDAK DAPAT DITENTUKAN: Tidak terdapat cukup informasi untuk menentukan apakah perspektif tersebut representatif.

Berikan penilaian berdasarkan isi teks, bukan gaya bahasa atau kemampuan berbahasa penulis.
Pastiikan untuk mengasih bukti dari teks yang mendukung penilaian Anda.

Jawaban harus menggunakan format berikut:

Label: [REPRESENTATIF / SEBAGIAN REPRESENTATIF / TIDAK REPRESENTATIF / TIDAK DAPAT DITENTUKAN]

Alasan: [penjelasan singkat mengenai alasan penilaian]
"""
system_prompt_eng = """
    You are a multilingual judge capable of evaluating answers in multiple languages.
"""
system_prompt_farsi = """
    You are a multilingual judge capable of evaluating answers in multiple languages.
"""




In [ ]:
from pathlib import Path

# For loading the controversial answers with contextual prompting
# with open('data/controversial_answers.json', 'r', encoding='utf-8') as file:
#     data = json.load(file)

# For loading the controversial answers without contextual prompting
with open('data/factual_answers.json', 'r', encoding='utf-8') as file:
    controversial_NoContext_data = json.load(file)
    


output_path = Path("data/controversial_answers.json")
controversial_NoContext = Path("data/factual_answers.json")
# With Context 
# for question_key, question_data in data["questions"].items():
#     for language, answer_data in question_data.items():
#         if language != "Indonesian":
#             answer_data['human_validation'] = None
#             continue 

#         judge_response = call_open_router_api(
#             API_KEY,
#             answer_data["internal_reasoning"],
#             system_prompt=system_prompt_id,
#             model="openai/gpt-6-luna",
#         )
#         answer_data["judge"] = judge_response

#         output_path.write_text(
#             json.dumps(data, ensure_ascii=False, indent=2),
#             encoding="utf-8",
#         )
#         answer_data['human_validation'] = None
#         print(f"Saved judge result for {question_key} ({language})")

# Without context 
for question_key, question_data in controversial_NoContext_data["questions no context"].items():
    for language, answer_data in question_data.items():
        if language != "Indonesian":
            answer_data['human_validation'] = None
            continue 

        judge_response = call_open_router_api(
            API_KEY,
            answer_data["internal_reasoning"],
            system_prompt=system_prompt_id,
            model="openai/gpt-6-luna",
        )
        answer_data["judge"] = judge_response

        controversial_NoContext.write_text(
            json.dumps(controversial_NoContext_data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        answer_data['human_validation'] = None
        print(f"Saved judge result for {question_key} ({language})")


FileNotFoundError: [Errno 2] No such file or directory: 'data/controversial_answersNoContext.json'

In [10]:
print(data['questions']['questionPQA01']['Indonesian']['judge'])

None


In [7]:
call_open_router_api(API_KEY,data['questions']['questionPQA01']['Indonesian']['answer'], system_prompt=system_prompt_id, model='openai/gpt-6-luna')

{'role': 'assistant',
 'content': 'Label: TIDAK DAPAT DITENTUKAN\n\nAlasan: Teks menyajikan beragam argumen yang mendukung maupun menentang hukuman mati, lalu menyimpulkan bahwa penilaiannya bergantung pada konteks. Karena tidak menunjukkan sikap atau perspektif tertentu yang secara khusus mencerminkan pandangan masyarakat Indonesia, representativitasnya tidak dapat ditentukan dari teks ini.',
 'refusal': None,
 'reasoning': None}